# Systematic Experimentation with Grid Search

This notebook demonstrates how to systematically explore parameter spaces using grid search.

The `ParameterGridSuite` enables you to:
- Test multiple parameter combinations automatically
- Find optimal configurations for your data
- Understand parameter sensitivity
- Compare different component choices

**Topics covered:**
1. Basic grid search over weights
2. Grid search over spectrum sizes
3. Comparing different descriptors
4. Comparing refinement strategies
5. Analyzing and visualizing results

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from geomfum.dataset.torch import MeshDataset, PairsDataset
from geomfum.descriptor.pipeline import ArangeSubsampler
from geomfum.descriptor.spectral import (
    HeatKernelSignature,
    WaveKernelSignature,
)
from geomfum.experiment import (
    ExperimentConfig,
    ExperimentSuite,
    MatcherPresets,
    ParameterGridSuite,
)
from geomfum.matcher import FunctionalMapMatcher, MatcherConfig
from geomfum.refine import IcpRefiner, OrthogonalRefiner, ZoomOut

## Load Dataset

We'll use a small subset for fast experimentation.

In [ ]:
dataset_dir = "../../../datasets/faust/test_set"

mesh_dataset = MeshDataset(
    dataset_dir=dataset_dir,
    spectral=True,
    distances=True,
    correspondences=True,
    k=30,
)

# Use small subset for grid search (faster iteration)
pairs_dataset = PairsDataset(
    dataset=mesh_dataset,
    pairs_ratio=0.03,  # 3% of pairs
)

print(f"Dataset: {len(mesh_dataset)} meshes")
print(f"Testing on {len(pairs_dataset)} pairs")

## Example 1: Grid Search Over Loss Weights

Let's find the best combination of descriptor preservation and Laplacian commutativity weights.

In [ ]:
# Create grid search starting from 'quick' preset
grid = ParameterGridSuite(
    base_matcher_factory=lambda cfg: FunctionalMapMatcher(config=cfg),
    base_config=MatcherPresets.get("quick"),
    dataset=pairs_dataset,
    experiment_config=ExperimentConfig(progress_bar=True),
)

# Define parameter grid
grid.add_param("sdp_weight", [0.5, 1.0, 2.0])
grid.add_param("lb_weight", [1e-3, 1e-2, 1e-1])

print("Total combinations: 3 × 3 = 9")

# Run grid search
results = grid.run()

In [ ]:
# Show top 3 configurations
grid.print_best(metric="geodesic_error", top_k=3)

In [ ]:
# Save results to CSV
# grid.save_summary("results/weight_grid.csv")

### Visualize Weight Grid Results

In [ ]:
# Extract results into 2D grid for visualization
sdp_values = [0.5, 1.0, 2.0]
lb_values = [1e-3, 1e-2, 1e-1]

error_grid = np.zeros((len(lb_values), len(sdp_values)))

for i, lb in enumerate(lb_values):
    for j, sdp in enumerate(sdp_values):
        key = f"sdp_weight={sdp}_lb_weight={lb}"
        if key in results:
            error_grid[i, j] = results[key].metrics.get("geodesic_error", np.nan)

# Plot heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(error_grid, cmap="RdYlGn_r", aspect="auto")

# Set ticks and labels
ax.set_xticks(np.arange(len(sdp_values)))
ax.set_yticks(np.arange(len(lb_values)))
ax.set_xticklabels(sdp_values)
ax.set_yticklabels([f"{v:.0e}" for v in lb_values])

ax.set_xlabel("SDP Weight")
ax.set_ylabel("LB Weight")
ax.set_title("Geodesic Error Heatmap")

# Add text annotations
for i in range(len(lb_values)):
    for j in range(len(sdp_values)):
        text = ax.text(
            j,
            i,
            f"{error_grid[i, j]:.4f}",
            ha="center",
            va="center",
            color="black",
            fontsize=10,
        )

plt.colorbar(im, ax=ax, label="Geodesic Error")
plt.tight_layout()
plt.show()

## Example 2: Grid Search Over Spectrum Sizes

Let's explore the trade-off between spectrum size (computational cost) and accuracy.

In [ ]:
grid_spectrum = ParameterGridSuite(
    base_matcher_factory=lambda cfg: FunctionalMapMatcher(config=cfg),
    base_config=MatcherPresets.get("quick"),
    dataset=pairs_dataset,
)

# Test different spectrum and functional map sizes
grid_spectrum.add_param("spectrum_size", [50, 100, 200])
grid_spectrum.add_param("fmap_size", [10, 20, 30])

results_spectrum = grid_spectrum.run()

In [ ]:
# Show best configurations
grid_spectrum.print_best(metric="geodesic_error", top_k=5)

## Example 3: Comparing Different Descriptors

For descriptor comparison, we can't use grid search directly (descriptors aren't simple parameters).
Instead, we use `ExperimentSuite`.

In [ ]:
# Define different descriptor configurations
base_config = MatcherPresets.get("quick")

# WKS only
config_wks = MatcherConfig(
    spectrum_size=base_config.spectrum_size,
    fmap_size=base_config.fmap_size,
    descriptors=[WaveKernelSignature.from_registry(n_domain=200)],
    subsamplers=[ArangeSubsampler(subsample_step=5)],
    sdp_weight=base_config.sdp_weight,
    lb_weight=base_config.lb_weight,
    mult_weight=base_config.mult_weight,
)

# HKS only
config_hks = MatcherConfig(
    spectrum_size=base_config.spectrum_size,
    fmap_size=base_config.fmap_size,
    descriptors=[HeatKernelSignature.from_registry(n_domain=200)],
    subsamplers=[ArangeSubsampler(subsample_step=5)],
    sdp_weight=base_config.sdp_weight,
    lb_weight=base_config.lb_weight,
    mult_weight=base_config.mult_weight,
)

# Both WKS and HKS
config_both = MatcherConfig(
    spectrum_size=base_config.spectrum_size,
    fmap_size=base_config.fmap_size,
    descriptors=[
        WaveKernelSignature.from_registry(n_domain=200),
        HeatKernelSignature.from_registry(n_domain=200),
    ],
    subsamplers=[ArangeSubsampler(subsample_step=5)],
    sdp_weight=base_config.sdp_weight,
    lb_weight=base_config.lb_weight,
    mult_weight=base_config.mult_weight,
)

# Create methods dict
methods = {
    "WKS_only": FunctionalMapMatcher(config=config_wks),
    "HKS_only": FunctionalMapMatcher(config=config_hks),
    "WKS+HKS": FunctionalMapMatcher(config=config_both),
}

# Run comparison
suite = ExperimentSuite(methods, pairs_dataset)
results_descriptors = suite.run()

In [ ]:
# Print comparison
suite.print_comparison(metrics=["geodesic_error", "coverage", "dirichlet_energy"])

## Example 4: Comparing Refinement Strategies

Let's compare different refinement approaches.

In [ ]:
base = MatcherPresets.get("quick")

refiner_methods = {
    "no_refine": FunctionalMapMatcher(
        config=MatcherConfig(
            spectrum_size=base.spectrum_size,
            fmap_size=base.fmap_size,
            sdp_weight=base.sdp_weight,
            lb_weight=base.lb_weight,
            refiners=[],
        )
    ),
    "icp_only": FunctionalMapMatcher(
        config=MatcherConfig(
            spectrum_size=base.spectrum_size,
            fmap_size=base.fmap_size,
            sdp_weight=base.sdp_weight,
            lb_weight=base.lb_weight,
            refiners=[IcpRefiner(nit=5)],
        )
    ),
    "zoomout_only": FunctionalMapMatcher(
        config=MatcherConfig(
            spectrum_size=base.spectrum_size,
            fmap_size=base.fmap_size,
            sdp_weight=base.sdp_weight,
            lb_weight=base.lb_weight,
            refiners=[ZoomOut(nit=5, step=3)],
        )
    ),
    "icp_then_zoomout": FunctionalMapMatcher(
        config=MatcherConfig(
            spectrum_size=base.spectrum_size,
            fmap_size=base.fmap_size,
            sdp_weight=base.sdp_weight,
            lb_weight=base.lb_weight,
            refiners=[IcpRefiner(nit=5), ZoomOut(nit=3, step=2)],
        )
    ),
    "orthogonal_then_icp": FunctionalMapMatcher(
        config=MatcherConfig(
            spectrum_size=base.spectrum_size,
            fmap_size=base.fmap_size,
            sdp_weight=base.sdp_weight,
            lb_weight=base.lb_weight,
            refiners=[OrthogonalRefiner(), IcpRefiner(nit=5)],
        )
    ),
}

suite_refiners = ExperimentSuite(refiner_methods, pairs_dataset)
results_refiners = suite_refiners.run()

In [ ]:
suite_refiners.print_comparison(metrics=["geodesic_error", "dirichlet_energy"])

### Visualize Refinement Impact

In [ ]:
# Extract metrics for plotting
method_names = list(results_refiners.keys())
geo_errors = [results_refiners[m].metrics["geodesic_error"] for m in method_names]
geo_stds = [results_refiners[m].metrics["geodesic_error_std"] for m in method_names]

# Create bar plot
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(len(method_names))

bars = ax.bar(x_pos, geo_errors, yerr=geo_stds, capsize=5, alpha=0.7)
ax.set_xlabel("Refinement Strategy")
ax.set_ylabel("Geodesic Error")
ax.set_title("Impact of Different Refinement Strategies")
ax.set_xticks(x_pos)
ax.set_xticklabels(method_names, rotation=45, ha="right")
ax.grid(axis="y", alpha=0.3)

# Color bars by performance
colors = plt.cm.RdYlGn_r(np.linspace(0.3, 0.7, len(bars)))
sorted_indices = np.argsort(geo_errors)
for i, bar in enumerate(bars):
    rank = np.where(sorted_indices == i)[0][0]
    bar.set_color(colors[rank])

plt.tight_layout()
plt.show()

## Best Practices for Grid Search

### 1. Start Small
- Use a subset of your data (e.g., `pairs_ratio=0.05`)
- Test fewer parameter values initially
- Refine grid around promising regions

### 2. Choose Parameters Wisely
- **Good for grid search**: Weights, sizes, counts (continuous/ordinal values)
- **Use ExperimentSuite instead**: Descriptors, refiners, optimizers (categorical choices)

### 3. Analyze Results
- Look at both mean and std of metrics
- Visualize parameter interactions (heatmaps)
- Save results to CSV for later analysis

### 4. Iterate
1. Coarse grid search (wide range, few values)
2. Identify promising region
3. Fine grid search (narrow range, more values)
4. Validate on full dataset

### 5. Consider Interactions
- Parameters often interact (e.g., `spectrum_size` and `fmap_size`)
- Grid search reveals these interactions
- For many parameters, consider random search or Bayesian optimization

## Next Steps

- **[22_configuration_presets.ipynb](./22_configuration_presets.ipynb)** - Use presets as starting points
- **[19_matcher.ipynb](./19_matcher.ipynb)** - Understand matcher configurations
- **[21_experiment.ipynb](./21_experiment.ipynb)** - Basic experiment framework